# CHISCo Text-Encoder Selection Benchmark

This notebook reproduces the **text-encoder selection benchmark** used in  
*Semantic Alignment of EEG and Text: A Contrastive Learning Framework for Decoding Imagined Speech*.

The goal is to identify a pretrained language embedding model that provides a well-structured and computationally practical semantic target space for subsequent EEG–language alignment. Six pretrained Chinese or multilingual sentence-embedding models are evaluated on the **39 semantic categories of the Chinese Imagined Speech Corpus (CHISCo)**.

For each candidate encoder, sentence embeddings are kept **frozen** and evaluated using four complementary approaches:

- **Logistic Regression (LR)** — linear probe of semantic separability
- **Multi-Layer Perceptron (MLP)** — nonlinear neural probe
- **XGBoost** — nonlinear tree-based probe
- **Zero-shot prototype matching** — cosine-similarity classification against embedded category names without supervised probe training

Evaluation uses **five-fold stratified cross-validation with sentence-level grouping**, ensuring that identical sentences cannot appear across training, validation, and held-out partitions. Within each outer fold, 15% of the training sentence groups are reserved for validation. The outer test partition is intentionally left unused in this benchmark, since the purpose of this stage is model selection rather than final EEG decoding evaluation.

Only the CHISCo `textdataset/` files are required. **No EEG recordings are loaded or processed in this notebook.**

The notebook:

1. Loads and validates the 39-category CHISCo text dataset.
2. Constructs reproducible sentence-grouped cross-validation splits.
3. Encodes the original Chinese stimulus sentences using six candidate pretrained language models.
4. Evaluates semantic category separability with LR, MLP, XGBoost, and zero-shot matching.
5. Aggregates validation accuracy as **mean ± standard deviation across five folds**.
6. Exports fold-level and summary results as CSV files.
7. Generates the language-model comparison figure used to guide text-encoder selection.

The benchmark compares:

- `BAAI/bge-small-zh-v1.5`
- `BAAI/bge-base-zh-v1.5`
- `BAAI/bge-large-zh-v1.5`
- `intfloat/multilingual-e5-base`
- `intfloat/multilingual-e5-large`
- `sentence-transformers/paraphrase-multilingual-mpnet-base-v2`

In the accompanying study, **BAAI/bge-small-zh-v1.5** was selected as the frozen text encoder because it offered strong semantic-category separability while retaining a compact **512-dimensional representation and approximately 24M parameters**, providing a favorable trade-off between performance and computational efficiency.

In [ ]:
# Run once if the environment is missing these packages.
%pip install -q sentence-transformers xgboost openpyxl tqdm

In [ ]:
import os
GLOBAL_SEED = 1337
os.environ["PYTHONHASHSEED"] = str(GLOBAL_SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc, random, time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.neural_network import MLPClassifier
from tqdm.auto import tqdm
from xgboost import XGBClassifier

random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

# Point this to any imported CHISCo dataset that contains textdataset/.
DATA_ROOT = Path(os.getenv(
    "CHISCO_DATA_ROOT",
    "/kaggle/input/datasets/shahryarnamdari/chisco-is-sub01/chisco_sub01_ready",
))
OUTPUT_DIR = Path(os.getenv(
    "CHISCO_BENCHMARK_OUTPUT_DIR",
    "/kaggle/working/chisco_text_encoder_benchmark" if Path("/kaggle").exists() else "./chisco_text_encoder_benchmark",
))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_FOLDS, VAL_RATIO_OF_TRAIN, NUM_CLASSES = 5, 0.15, 39

chinese_to_english = {
    "预订和旅行安排": "Travel Arrangements",
    "住房和设施": "Housing and Facilities",
    "自然和天气": "Nature and Weather",
    "个人行为和日常活动": "Personal Behavior and Daily Activities",
    "金融和付款": "Finance",
    "饮食和用餐": "Food and Dining",
    "旅行和行李管理": "Travel Affairs",
    "交通和出行": "Transportation and Commuting",
    "旅游和度假": "Vacation",
    "设备故障和环境问题": "Equipment Malfunction or Environmental Issues",
    "价格和费用": "Prices and Costs",
    "时间和日程安排": "Time and Scheduling",
    "衣物和服饰": "Clothing",
    "语言和学习": "Learning",
    "欢迎和感谢": "Welcoming and Thanking",
    "道歉和请求原谅": "Apologies",
    "个人信息": "Personal Information",
    "询问和个人事务": "Inquiries and Personal Matters",
    "饮食习惯": "Eating Habits",
    "产品和质量保证": "Products and Quality Assurance",
    "健康和安全建议": "Health and Safety",
    "家庭关系和家庭事件": "Family Relationships and Events",
    "提供和请求帮助": "Providing and Requesting Assistance",
    "理发和美容护理": "Hairdressing and Beauty Care",
    "节日和庆祝活动": "Festivals and Celebrations",
    "互联网和信息技术": "Internet and Information Technology",
    "健康和身体不适": "Physical Discomfort",
    "情感和人际关系": "Emotions and Interpersonal Relationships",
    "社交和聚会活动": "Gathering Activities",
    "人际交往和情感表达": "Social Interactions",
    "问候和情感状态": "Greetings",
    "工作和职场交流": "Work",
    "表演艺术": "Performing Arts",
    "教育和学习": "Education",
    "娱乐和媒体消费": "Entertainment and Media Consumption",
    "求职和职业发展": "Job Hunting and Career Development",
    "健身": "Fitness",
    "兴趣爱好": "Hobbies",
    "体育和运动": "Sports",
}
CLASS_PHRASES_ZH = list(chinese_to_english)
CLASS2ID = {label: i for i, label in enumerate(CLASS_PHRASES_ZH)}
assert len(CLASS2ID) == NUM_CLASSES

CANDIDATES = [
    "BAAI/bge-small-zh-v1.5",
    "BAAI/bge-base-zh-v1.5",
    "BAAI/bge-large-zh-v1.5",
    "intfloat/multilingual-e5-base",
    "intfloat/multilingual-e5-large",
    "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
]
MODEL_META = {
    "BAAI/bge-small-zh-v1.5": ("bge-small-zh", 512, 24),
    "BAAI/bge-base-zh-v1.5": ("bge-base-zh", 768, 102),
    "BAAI/bge-large-zh-v1.5": ("bge-large-zh", 1024, 326),
    "intfloat/multilingual-e5-base": ("mE5-base", 768, 278),
    "intfloat/multilingual-e5-large": ("mE5-large", 1024, 560),
    "sentence-transformers/paraphrase-multilingual-mpnet-base-v2": ("mpnet-multi-v2", 768, 278),
}
METHODS = ["LR", "MLP", "XGBoost", "Zero-shot"]
XGB_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Data root:", DATA_ROOT)
print("Seed:", GLOBAL_SEED, "| XGBoost:", XGB_DEVICE)
print("Output:", OUTPUT_DIR)

## Load text data and reproduce the CV splits

Only Excel files inside `textdataset/` are used. Sentences are treated as groups, so the same sentence cannot occur on both sides of a split.

In [ ]:
def find_textdataset(root):
    root = Path(root)
    if root.name == "textdataset" and root.is_dir():
        return root
    direct = root / "textdataset"
    if direct.is_dir():
        return direct
    matches = sorted(path for path in root.rglob("textdataset") if path.is_dir())
    if not matches:
        raise FileNotFoundError(f"No textdataset/ folder found under {root}")
    return matches[0]

def load_text_data(text_dir):
    files = sorted(Path(text_dir).glob("*.xlsx"))
    if not files:
        raise FileNotFoundError(f"No .xlsx files found in {text_dir}")
    frames = [pd.read_excel(path) for path in files]
    frames = [frame for frame in frames if frame.shape[1] >= 2]
    if not frames:
        raise RuntimeError("No valid textdataset Excel files were found")
    data = pd.concat(frames, ignore_index=True)
    columns = [str(column) for column in data.columns]
    def find_column(names, fallback):
        return next((col for name in names for col in columns if name.lower() in col.lower()), columns[fallback])
    sentence_col = find_column(["句子", "sentence", "text"], 0)
    label_col = find_column(["标签", "label", "class"], 1)
    data = data[[sentence_col, label_col]].rename(columns={sentence_col: "sentence_zh", label_col: "label_zh"})
    data = data.dropna().astype(str)
    data["sentence_zh"] = data["sentence_zh"].str.strip()
    data["label_zh"] = data["label_zh"].str.strip()
    data = data[data["label_zh"].isin(CLASS2ID)].copy()
    if (data.groupby("sentence_zh")["label_zh"].nunique() > 1).any():
        raise ValueError("At least one sentence maps to multiple semantic labels")
    groups = data.groupby("sentence_zh", sort=True, as_index=False).agg(label_zh=("label_zh", "first"))
    groups["label_en"] = groups["label_zh"].map(chinese_to_english)
    groups["label_id"] = groups["label_zh"].map(CLASS2ID).astype(np.int64)
    groups["group_id"] = np.arange(len(groups), dtype=np.int64)
    return groups, len(data), len(files)

def build_cv_splits(groups):
    group_ids = groups["group_id"].to_numpy(np.int64)
    labels = groups["label_id"].to_numpy(np.int64)
    if np.bincount(labels, minlength=NUM_CLASSES).min() < N_FOLDS:
        raise ValueError(f"Each class needs at least {N_FOLDS} sentence groups")
    outer = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=GLOBAL_SEED)
    splits = []
    for fold, (trainval_pos, test_pos) in enumerate(outer.split(group_ids, labels)):
        trainval_ids, test_ids = group_ids[trainval_pos], group_ids[test_pos]
        trainval_labels = labels[trainval_pos]
        inner = StratifiedShuffleSplit(n_splits=1, test_size=VAL_RATIO_OF_TRAIN, random_state=GLOBAL_SEED)
        train_rel, val_rel = next(inner.split(trainval_ids, trainval_labels))
        splits.append({
            "fold": fold,
            "train_ids": trainval_ids[train_rel],
            "val_ids": trainval_ids[val_rel],
            "test_ids": test_ids,
        })
    return splits

def validate_splits(groups, splits):
    all_ids = np.arange(len(groups), dtype=np.int64)
    test_ids = []
    for split in splits:
        train, val, test = split["train_ids"], split["val_ids"], split["test_ids"]
        if any(np.intersect1d(a, b).size for a, b in ((train, val), (train, test), (val, test))):
            raise AssertionError(f"Fold {split['fold']} contains overlapping sentence groups")
        if not np.array_equal(np.sort(np.concatenate([train, val, test])), all_ids):
            raise AssertionError(f"Fold {split['fold']} does not cover every sentence group")
        test_ids.append(test)
    if not np.array_equal(np.sort(np.concatenate(test_ids)), all_ids):
        raise AssertionError("Every sentence group must occur in exactly one outer test fold")

TEXTDATASET_DIR = find_textdataset(DATA_ROOT)
sentence_groups, n_rows, n_files = load_text_data(TEXTDATASET_DIR)
CV_SPLITS = build_cv_splits(sentence_groups)
validate_splits(sentence_groups, CV_SPLITS)

split_summary = pd.DataFrame([{
    "fold": s["fold"] + 1,
    "train": len(s["train_ids"]),
    "validation": len(s["val_ids"]),
    "held-out test (unused)": len(s["test_ids"]),
} for s in CV_SPLITS])
print(f"Loaded {n_rows} valid rows from {n_files} Excel files")
print(f"Sentence groups: {len(sentence_groups)} | Classes: {sentence_groups['label_id'].nunique()}")
print("Text dataset:", TEXTDATASET_DIR)
display(split_summary)

## Benchmark frozen text encoders

For each fold, only its training and validation sentences are encoded. The held-out outer test fold is not encoded or scored. Probe hyperparameters are kept consistent with the original benchmark.

In [ ]:
def encode_texts(model, texts):
    return model.encode(list(texts), normalize_embeddings=True, show_progress_bar=False).astype(np.float32)

def make_probes():
    return {
        "LR": LogisticRegression(max_iter=1000, random_state=GLOBAL_SEED, n_jobs=-1),
        "MLP": MLPClassifier(
            hidden_layer_sizes=(512, 256), activation="relu", solver="adam", alpha=1e-4,
            batch_size=128, learning_rate_init=1e-3, max_iter=200, early_stopping=True,
            n_iter_no_change=5, validation_fraction=0.1, random_state=GLOBAL_SEED,
        ),
        "XGBoost": XGBClassifier(
            n_estimators=200, max_depth=5, learning_rate=0.1, subsample=0.9, colsample_bytree=0.9,
            objective="multi:softmax", num_class=NUM_CLASSES, tree_method="hist", device=XGB_DEVICE,
            eval_metric="mlogloss", random_state=GLOBAL_SEED, n_jobs=-1,
        ),
    }

def evaluate_fold(model, class_embeddings, split):
    train = sentence_groups.iloc[split["train_ids"]]
    val = sentence_groups.iloc[split["val_ids"]]
    X_train, X_val = encode_texts(model, train["sentence_zh"]), encode_texts(model, val["sentence_zh"])
    y_train = train["label_id"].to_numpy(np.int64)
    y_val = val["label_id"].to_numpy(np.int64)
    scores = {}
    for name, probe in make_probes().items():
        probe.fit(X_train, y_train)
        scores[name] = accuracy_score(y_val, probe.predict(X_val))
    zero_shot = np.argmax(cosine_similarity(X_val, class_embeddings), axis=1)
    scores["Zero-shot"] = accuracy_score(y_val, zero_shot)
    del X_train, X_val
    gc.collect()
    return scores

def benchmark_model(model_name):
    start = time.time()
    model = SentenceTransformer(model_name)
    class_embeddings = encode_texts(model, CLASS_PHRASES_ZH)
    rows = []
    for split in tqdm(CV_SPLITS, desc=MODEL_META[model_name][0], leave=False):
        scores = evaluate_fold(model, class_embeddings, split)
        rows.extend({"model": model_name, "fold": split["fold"] + 1, "method": method, "accuracy": acc}
                    for method, acc in scores.items())
    del model, class_embeddings
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"{MODEL_META[model_name][0]}: {(time.time() - start) / 60:.1f} min")
    return rows

rows = []
for model_name in tqdm(CANDIDATES, desc="Language models"):
    rows.extend(benchmark_model(model_name))
fold_results = pd.DataFrame(rows)
fold_results.to_csv(OUTPUT_DIR / "text_encoder_fold_results.csv", index=False)
display(fold_results.head(12))

## Five-fold mean ± SD

In [ ]:
summary = fold_results.groupby(["model", "method"], sort=False)["accuracy"].agg(
    mean="mean", std=lambda x: x.std(ddof=1)
).reset_index()
summary["model"] = pd.Categorical(summary["model"], CANDIDATES, ordered=True)
summary["method"] = pd.Categorical(summary["method"], METHODS, ordered=True)
summary = summary.sort_values(["model", "method"]).reset_index(drop=True)
summary.to_csv(OUTPUT_DIR / "text_encoder_cv_summary.csv", index=False)
display(summary)

colors = {"LR": "#577E89", "MLP": "#6F9F9C", "XGBoost": "#E1A36F", "Zero-shot": "#DEC484"}
x, width = np.arange(len(CANDIDATES)), 0.20
fig, ax = plt.subplots(figsize=(7.15, 4.4))

for i, method in enumerate(METHODS):
    v = summary[summary["method"] == method].set_index("model").reindex(CANDIDATES)
    means, stds = v["mean"].to_numpy(float), v["std"].to_numpy(float)
    pos = x + (i - 1.5) * width
    bars = ax.bar(pos, means, width, yerr=stds, capsize=2.5, color=colors[method], label=method)
    for bar, mean, std in zip(bars, means, stds):
        ax.text(bar.get_x() + bar.get_width()/2, mean + std + 0.012, f"{mean:.2f}",
                ha="center", va="bottom", fontsize=7)

labels = [f"{MODEL_META[m][0]}\n{MODEL_META[m][2]}M, dim={MODEL_META[m][1]}" for m in CANDIDATES]
ax.set(xticks=x, xticklabels=labels, ylabel="Validation accuracy", ylim=(0, 1.0))
plt.setp(ax.get_xticklabels(), rotation=18, ha="right")
ax.set_title("Comparison of Language Models and Classifiers", fontsize=10.5, pad=8)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.11), ncol=4, frameon=False)
ax.grid(axis="y", alpha=0.2)

fig.subplots_adjust(top=0.83, bottom=0.25, left=0.10, right=0.98)
fig.savefig(OUTPUT_DIR / "lm_classifier_comparison_cv.pdf", bbox_inches="tight")
fig.savefig(OUTPUT_DIR / "lm_classifier_comparison_cv.png", dpi=400, bbox_inches="tight")
plt.show()
print("Saved results to", OUTPUT_DIR.resolve())